In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt

# --- 1. Configuration ---
SYMBOL = 'ETHUSDT'
MODEL_TYPE = 'lstm'
EPOCHS = 50
BATCH_SIZE = 64

# --- 2. Local VS Code Pathing ---
BASE_DIR = os.path.abspath(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, f'processed_data_{MODEL_TYPE}', SYMBOL)
MODEL_SAVE_PATH = os.path.join(BASE_DIR, f'{MODEL_TYPE}_model_{SYMBOL}.keras')

print(f"📁 Loading {SYMBOL} data from:\n{DATA_DIR}")

# --- 3. Load Data ---
try:
    X_train = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
    y_train = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
    X_val = np.load(os.path.join(DATA_DIR, 'X_val.npy'))
    y_val = np.load(os.path.join(DATA_DIR, 'y_val.npy'))
    X_test = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
    y_test = np.load(os.path.join(DATA_DIR, 'y_test.npy'))
    print("✅ Data successfully loaded!")
    print(f"X_train shape: {X_train.shape}")
except FileNotFoundError as e:
    print(f"❌ Error loading data: {e}\nPlease run the preprocessing script for {SYMBOL} first.")

# --- 4. Build LSTM Architecture ---
def build_lstm(input_shape):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1, activation='linear') # Linear activation for regression (price/return prediction)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

model = build_lstm((X_train.shape[1], X_train.shape[2]))
model.summary()

# --- 5. Callbacks & Training ---
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_loss', save_best_only=True)
]

print(f"\n🚀 Starting training for {SYMBOL}...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# --- 6. Plot Training History ---
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title(f'{SYMBOL} LSTM Training History')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.show()

# --- 7. Evaluation ---
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"🎯 Test MAE: {test_mae:.5f}")
print(f"💾 Best model saved to: {MODEL_SAVE_PATH}")